This notebook is a restartable Google Colab workflow.

The embedded source snapshot is identified by its Git commit and SHA-256.
Drive markers and hashes control resume behavior. Full work starts only after the tiny validation gate.


In [ ]:
# [RH-BOOTSTRAP] Drive, source identity, provenance, and resumability
from __future__ import annotations

import base64
import hashlib
import importlib.metadata as importlib_metadata
import io
import json
import os
from pathlib import Path, PurePosixPath
import platform
import shutil
import subprocess
import sys
import tarfile
import time

# Full Colab runs use two numerical threads.  This also keeps tiny validation
# deterministic when a runtime exposes a large CPU pool.
for _name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS", "RAYON_NUM_THREADS"):
    os.environ[_name] = "2"
os.environ.setdefault("PYTHONHASHSEED", "0")

SOURCE_COMMIT = "44bae4c19206a223d4cc9e5f1825fe7de5bc75e4"
SOURCE_DIRTY = True
SOURCE_ARCHIVE_SHA256 = "8d3b6c7dae0445213aaacbe214feae78c59b1de6c1cba9499b5d1b745a942ed2"
SOURCE_ARCHIVE_B64 = "..."
GENERATOR_VERSION = "2026-08-26.2"
EXPERIMENT = "generic_mlp"
CONFIG_NAME = "generic_colab.toml"
REQUIREMENTS_NAME = "requirements-colab.txt"
RUN_ID = os.environ.get("LRH_RUN_ID", f"{EXPERIMENT}-{SOURCE_ARCHIVE_SHA256[:12]}")
REMOTE_ROOT = Path("/content/drive/MyDrive/two_equilibria/v1")
REMOTE_RUN_DIR = REMOTE_ROOT / "runs" / EXPERIMENT / RUN_ID
REMOTE_MARKER_DIR = REMOTE_RUN_DIR / "markers"
WORK_DIR = Path("/content/rh_work") / RUN_ID
SOURCE_ROOT = WORK_DIR / "source"


def _sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_bytes(path: Path, payload: bytes) -> None:
    """Write a file safely, including files on a mounted Drive filesystem."""

    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".partial.{os.getpid()}")
    temporary.write_bytes(payload)
    with temporary.open("rb") as handle:
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def atomic_write_json(path: Path, value: object) -> None:
    atomic_write_bytes(
        path,
        (json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False) + "\n").encode("utf-8"),
    )


def atomic_copy_to_drive(source: Path, destination: Path) -> None:
    """Copy a completed local artifact to Drive before exposing its final name."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + f".partial.{os.getpid()}")
    shutil.copy2(source, temporary)
    os.replace(temporary, destination)


def _safe_extract(payload: bytes, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(payload), mode="r:*") as archive:
        members = archive.getmembers()
        for member in members:
            path = PurePosixPath(member.name)
            if path.is_absolute() or ".." in path.parts:
                raise RuntimeError(f"unsafe source archive member: {member.name}")
        archive.extractall(destination)


def materialize_source() -> None:
    payload = base64.b64decode(SOURCE_ARCHIVE_B64.encode("ascii"))
    if _sha256_bytes(payload) != SOURCE_ARCHIVE_SHA256:
        raise RuntimeError("embedded source archive hash mismatch")
    if SOURCE_ROOT.exists():
        existing = list(SOURCE_ROOT.rglob("*"))
        if existing:
            sys.path.insert(0, str(SOURCE_ROOT / "src"))
            return
    _safe_extract(payload, SOURCE_ROOT)
    sys.path.insert(0, str(SOURCE_ROOT / "src"))


def _drive_mount() -> None:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("This notebook must run in Google Colab") from exc
    drive.mount("/content/drive", force_remount=False)


def _runtime_info() -> dict[str, object]:
    info: dict[str, object] = {
        "python": sys.version,
        "platform": platform.platform(),
        "machine": platform.machine(),
        "cpu_count": os.cpu_count(),
        "runtime_marker": os.environ.get("COLAB_RELEASE_TAG", "unknown"),
        "accelerator": {"available": False, "name": None, "memory_bytes": None},
    }
    try:
        import psutil

        info["host_memory_bytes"] = psutil.virtual_memory().total
    except ImportError:
        info["host_memory_bytes"] = None
    try:
        import torch

        accelerator = info["accelerator"]
        assert isinstance(accelerator, dict)
        accelerator["available"] = bool(torch.cuda.is_available())
        if accelerator["available"]:
            device = torch.cuda.current_device()
            properties = torch.cuda.get_device_properties(device)
            accelerator["name"] = properties.name
            accelerator["memory_bytes"] = properties.total_memory
            accelerator["cuda"] = torch.version.cuda
        info["torch"] = torch.__version__
    except ImportError:
        info["torch"] = None
    return info


def _required_versions(requirements: Path) -> dict[str, str]:
    expected: dict[str, str] = {}
    for line in requirements.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("-r "):
            expected.update(_required_versions(requirements.parent / line[3:].strip()))
            continue
        if line.startswith("-") or "==" not in line:
            continue
        name, version = line.split("==", 1)
        expected[name.lower().replace("-", "_")] = version
    return expected


def assert_pinned_versions(requirements: Path) -> dict[str, str]:
    expected = _required_versions(requirements)
    observed: dict[str, str] = {}
    mismatches: list[str] = []
    for normalized, wanted in expected.items():
        try:
            actual = importlib_metadata.version(normalized)
        except importlib_metadata.PackageNotFoundError:
            try:
                actual = importlib_metadata.version(normalized.replace("_", "-"))
            except importlib_metadata.PackageNotFoundError:
                actual = "missing"
        observed[normalized] = actual
        if actual != wanted:
            mismatches.append(f"{normalized}={actual}, expected {wanted}")
    if mismatches:
        raise RuntimeError("pinned dependency mismatch: " + "; ".join(mismatches))
    return observed


def configured_seeds(config_name: str = CONFIG_NAME) -> dict[str, object]:
    import tomllib

    raw = tomllib.loads(config_path(config_name).read_text(encoding="utf-8"))
    return {
        str(key): value
        for key, value in raw.items()
        if "seed" in str(key).lower() or str(key).lower().endswith("seeds")
    }


def install_and_record_versions() -> None:
    requirements = SOURCE_ROOT / REQUIREMENTS_NAME
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", "-r", str(requirements)],
        check=True,
    )
    observed = assert_pinned_versions(requirements)
    atomic_write_json(
        REMOTE_RUN_DIR / "provenance" / "packages.json",
        {"requirements": REQUIREMENTS_NAME, "requirements_sha256": sha256_file(requirements), "packages": observed},
    )


def config_path(config_name: str = CONFIG_NAME) -> Path:
    path = SOURCE_ROOT / "configs" / config_name
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def config_identity(config_name: str = CONFIG_NAME) -> dict[str, object]:
    path = config_path(config_name)
    return {
        "name": config_name,
        "path": str(path),
        "sha256": sha256_file(path),
        "config_sha256": config_run_sha256(config_name),
        "bytes": path.stat().st_size,
    }


def config_run_sha256(config_name: str = CONFIG_NAME) -> str:
    """Hash the validated config shape used by the campaign completion marker."""

    import tomllib

    raw = tomllib.loads(config_path(config_name).read_text(encoding="utf-8"))
    labels = {"c_on_min": 0.95, "invariant_c_off_min": 0.90, "strategic_c_off_max": 0.10}
    labels.update(raw.get("labels", {}))
    statistics = {
        "dip_bootstrap": 2000,
        "mixture_bootstrap": 2000,
        "bootstrap_seed": 8675309,
        "alpha": 0.05,
        "minimum_component_weight": 0.10,
        "minimum_gap_separation": 0.30,
        "bic_delta": 10.0,
    }
    statistics.update(raw.get("statistics", {}))
    validated = dict(raw)
    validated["labels"] = labels
    validated["statistics"] = statistics
    return _sha256_bytes(
        json.dumps(validated, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False).encode(
            "utf-8"
        )
    )


def config_experiment(config_name: str = CONFIG_NAME) -> str:
    import tomllib

    value = tomllib.loads(config_path(config_name).read_text(encoding="utf-8")).get("experiment")
    if not isinstance(value, str) or not value:
        raise RuntimeError(f"config {config_name} has no experiment name")
    return value


def config_completed(config_name: str) -> bool:
    config_run_dir = REMOTE_ROOT / "runs" / config_experiment(config_name) / RUN_ID
    return any(_valid_complete_marker(path, config_name) for path in _complete_marker_paths(config_run_dir))


def record_provenance(config_name: str = CONFIG_NAME, seeds: object = None) -> None:
    requirements = SOURCE_ROOT / REQUIREMENTS_NAME
    package_versions = assert_pinned_versions(requirements)
    provenance = {
        "schema_version": 1,
        "generator_version": GENERATOR_VERSION,
        "source_commit": SOURCE_COMMIT,
        "source_dirty": SOURCE_DIRTY,
        "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
        "source_files_embedded": True,
        "config": config_identity(config_name),
        "requirements": {"name": REQUIREMENTS_NAME, "sha256": sha256_file(requirements)},
        "packages": package_versions,
        "seed": seeds,
        "runtime": _runtime_info(),
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    atomic_write_json(REMOTE_RUN_DIR / "provenance" / "provenance.json", provenance)


def marker(name: str) -> Path:
    return REMOTE_MARKER_DIR / name


def _complete_marker_candidates(directory: Path) -> tuple[Path, ...]:
    return (directory / "completed.json", directory / "RUN_COMPLETE.json", directory / "run_complete.json")


def _complete_marker_paths(directory: Path) -> tuple[Path, ...]:
    return _complete_marker_candidates(directory) + _complete_marker_candidates(directory / "markers")


def _config_names(config_name: str | Iterable[str] | None) -> tuple[str, ...]:
    if config_name is None:
        return (CONFIG_NAME,)
    if isinstance(config_name, str):
        return (config_name,)
    names = tuple(str(name) for name in config_name)
    if not names or len(set(names)) != len(names):
        return ()
    return names


def _config_identity_matches(candidate: object, expected: dict[str, object]) -> bool:
    if isinstance(candidate, str):
        return candidate in {expected["sha256"], expected["config_sha256"]}
    if not isinstance(candidate, dict):
        return False
    return (
        candidate.get("name") == expected["name"]
        and candidate.get("sha256") == expected["sha256"]
        and candidate.get("config_sha256", expected["config_sha256"]) == expected["config_sha256"]
        and candidate.get("bytes") == expected["bytes"]
    )


def _marker_config_matches(value: dict[str, object], config_name: str | Iterable[str] | None) -> bool:
    names = _config_names(config_name)
    if not names:
        return False
    try:
        expected = {name: config_identity(name) for name in names}
    except (FileNotFoundError, OSError):
        return False

    # Combined notebook markers carry one identity per campaign config.  The
    # exact set prevents a stale marker from silently omitting one campaign.
    identities = value.get("config_identities")
    if len(names) > 1:
        if not isinstance(identities, dict) or set(identities) != set(names):
            return False
        if any(not _config_identity_matches(identities.get(name), expected[name]) for name in names):
            return False
        for key in ("config_sha256", "config_identity", "config"):
            if key in value and not _config_identity_matches(value[key], expected[names[0]]):
                return False
        configs = value.get("configs")
        if configs is not None and (not isinstance(configs, list) or set(configs) != set(names)):
            return False
        return True

    name = names[0]
    if isinstance(identities, dict):
        if set(identities) != {name} or not _config_identity_matches(identities.get(name), expected[name]):
            return False

    found = False
    for key in ("config_sha256", "config_identity", "config"):
        if key not in value:
            continue
        found = True
        if not _config_identity_matches(value[key], expected[name]):
            return False
    return found


def _source_identity_matches(value: dict[str, object]) -> bool:
    source_archive = value.get("source_archive_sha256")
    source_identity = value.get("source_identity")
    if source_archive is None and source_identity is None:
        return False
    if source_archive is not None and source_archive != SOURCE_ARCHIVE_SHA256:
        return False
    if source_identity is not None and source_identity != SOURCE_ARCHIVE_SHA256:
        return False
    return True


def _valid_complete_marker(path: Path, config_name: str | Iterable[str] | None = CONFIG_NAME) -> bool:
    if not path.is_file():
        return False
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(value, dict):
            return False
        if (
            value.get("state") == "complete"
            and path.name != "RUN_COMPLETE.json"
            and _source_identity_matches(value)
            and _marker_config_matches(value, config_name)
        ):
            return True
        # CheckpointStore's run marker binds completion through the run id,
        # checkpoint reference, and configuration identity.
        names = _config_names(config_name)
        if len(names) != 1 or not _source_identity_matches(value) or not _marker_config_matches(value, names):
            return False
        expected = config_identity(names[0])
        checkpoint = value.get("checkpoint")
        return (
            path.name == "RUN_COMPLETE.json"
            and value.get("run_id") == RUN_ID
            and value.get("config_identity") == expected["config_sha256"]
            and isinstance(checkpoint, dict)
            and checkpoint.get("run_id") == RUN_ID
            and checkpoint.get("config_identity") == expected["config_sha256"]
            and checkpoint.get("source_identity") == SOURCE_ARCHIVE_SHA256
        )
    except (FileNotFoundError, OSError, ValueError, TypeError, KeyError):
        return False


def completed(name: str, config_name: str | Iterable[str] | None = CONFIG_NAME) -> bool:
    if name in {"completed.json", "RUN_COMPLETE.json", "run_complete.json"}:
        return any(_valid_complete_marker(path, config_name) for path in _complete_marker_paths(REMOTE_RUN_DIR))
    return _valid_complete_marker(marker(name), config_name)


def write_marker(name: str, payload: dict[str, object], config_name: str = CONFIG_NAME) -> None:
    identity = config_identity(config_name)
    value = {
        **payload,
        "state": "complete",
        "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
        "config_sha256": identity["config_sha256"],
        "config_identity": identity,
    }
    atomic_write_json(marker(name), value)


def existing_outputs() -> dict[str, bool]:
    state = {"validation": completed("validation.done.json"), "run": completed("completed.json"), "export": completed("export.done.json")}
    print("resume state:", json.dumps(state, sort_keys=True))
    return state


def run_cli(command: str, *arguments: str) -> None:
    environment = os.environ.copy()
    environment["LRH_RUNTIME"] = "colab"
    environment["LRH_RUN_ID"] = RUN_ID
    environment["RH_SOURCE_COMMIT"] = SOURCE_COMMIT
    environment["RH_SOURCE_ARCHIVE_SHA256"] = SOURCE_ARCHIVE_SHA256
    environment["PYTHONPATH"] = str(SOURCE_ROOT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
    command_line = [sys.executable, "-m", "lean_reward_hacking.cli", command, *map(str, arguments)]
    subprocess.run(command_line, cwd=str(SOURCE_ROOT), env=environment, check=True)


ALLOWLISTED_BUNDLE_FILES = frozenset({
    "manifest.json",
    "bundle_manifest.json",
    "provenance.json",
    "checksums.sha256",
    "runs.csv",
    "pair_counts.csv",
    "checkpoint_metrics.csv",
    "final_summary.csv",
    "basin_cells.csv",
    "perturbation_trajectory.csv",
    "stats.json",
})
FORBIDDEN_BUNDLE_PARTS = frozenset({"raw", "checkpoints", "logs", "cache", "weights", "samples"})


def validate_compact_bundle(bundle: Path) -> None:
    if not bundle.is_dir():
        raise RuntimeError(f"compact bundle is missing: {bundle}")
    for path in bundle.rglob("*"):
        if not path.is_file():
            continue
        relative = path.relative_to(bundle).as_posix()
        if any(part in FORBIDDEN_BUNDLE_PARTS for part in PurePosixPath(relative).parts):
            raise RuntimeError(f"raw artifact leaked into compact bundle: {relative}")
        if relative not in ALLOWLISTED_BUNDLE_FILES:
            raise RuntimeError(f"unallowlisted compact-bundle file: {relative}")
    manifest = bundle / "manifest.json"
    if not manifest.is_file():
        raise RuntimeError("compact bundle has no manifest.json")


def export_compact_bundle() -> Path:
    local_bundle = Path("/content/rh_compact_bundle") / EXPERIMENT
    run_cli("export", "--remote-root", str(REMOTE_ROOT), "--local-bundle", str(local_bundle))
    validate_compact_bundle(local_bundle)
    write_marker("export.done.json", {"bundle": str(local_bundle), "bundle_sha256": sha256_file(local_bundle / "manifest.json")})
    print("compact bundle:", local_bundle)


_drive_mount()
materialize_source()
REMOTE_RUN_DIR.mkdir(parents=True, exist_ok=True)
REMOTE_MARKER_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# [RH-PACKAGES] Install the checked-in lock verbatim and assert every version.
install_and_record_versions()


In [ ]:
# [RH-PROVENANCE] Runtime, accelerator, package, seed, config, and source identity.
record_provenance(CONFIG_NAME, seeds=configured_seeds(CONFIG_NAME))
print(json.dumps(_runtime_info(), indent=2, sort_keys=True))


In [ ]:
# [RH-TINY-GATE] Required before any full run.
if not completed("validation.done.json", "generic_colab.toml"):
    run_cli("tiny-validate", "--config", str(config_path("generic_colab.toml")), "--remote-root", str(REMOTE_ROOT))
    write_marker("validation.done.json", {"config": config_identity("generic_colab.toml"), "gate": {"seed": 0, "updates": 2, "episodes_per_update": 2, "paired_eval_count": 4, "checkpoint_every": 1, "basin_grid": [1, 1], "perturbations": 1}}, config_name="generic_colab.toml")
else:
    print("tiny validation marker is valid; skipping the gate")


The generic control receives the same episode fields and reward. Its plain MLP has no named goal or oversight-gate modules. Audit-cue swaps and ablations are recorded by the project API.


In [ ]:
# [RH-FULL-RUN] The CLI reads sharding, seeds, device, and resume policy from each TOML.
existing_outputs()
if not config_completed("generic_colab.toml"):
    run_cli("colab-run", "--config", str(config_path("generic_colab.toml")), "--remote-root", str(REMOTE_ROOT))
else:
    print("generic_colab.toml completed marker is valid; skipping full work")
if not all(config_completed(name) for name in ['generic_colab.toml']):
    raise RuntimeError("one or more Colab run markers are incomplete after colab-run")
if not completed("completed.json", ['generic_colab.toml']):
    write_marker("completed.json", {"configs": ['generic_colab.toml'], "config_identities": {name: config_identity(name) for name in ['generic_colab.toml']}}, config_name='generic_colab.toml')
else:
    print("completed marker is valid; skipping full work")


In [ ]:
# [RH-EXPORT] Export only the allowlisted compact bundle.
if completed("export.done.json", CONFIG_NAME):
    print("export marker is valid; skipping export")
else:
    export_compact_bundle()
